Create vector embeddings of the sentences using SentenceTransformers

In [1]:
# Create sentence transformer using retrained minilm model
from local_utilities.directory import get_minilm_directory
from sentence_transformers import SentenceTransformer
encoder = SentenceTransformer(get_minilm_directory())

[nltk_data] Downloading package punkt_tab to /home/tobias/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
/home/tobias/.pyenv/versions/genai/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1738.28it/s, Materializing param=pooler.dense.weight]                             


In [2]:
# Get GPU
from local_utilities.gpu import get_gpu
device = get_gpu()

In [3]:
# Define encoding function that allows us encoding the
# sentences in batches without running out of GPU memory
import torch
from torch.nn.functional import normalize
from tqdm import tqdm

def encode_batch(sentences, encoder: SentenceTransformer, batch_size=8, device="cuda"):
    all_embeddings = []

    # Split whole batch into smaller batches
    for i in tqdm(range(0, len(sentences), batch_size)):
        current_batch = sentences[i:i+batch_size]

        # Encode batch
        batch_embeddings = encoder.encode(
            current_batch,
            convert_to_tensor=True,
            device=device
        )

        # Normalize embeddings (must be done on custom model)
        batch_embeddings = normalize(batch_embeddings, p=2, dim=1)
        
        # Move batch to CPU
        all_embeddings.append(batch_embeddings.cpu().float())

        # Free GPU memory
        del batch_embeddings
        if device.startswith("cuda"):
            torch.cuda.empty_cache()

    # Return all embeddings
    return torch.cat(all_embeddings, dim=0)

In [4]:
# Function that encodes a dataset
import pickle

def encode_dataset(human_texts, ai_texts, path, encoder, batch_size=8, device="cuda"):
    # Encode AI texts
    ai_data = encode_batch(ai_texts, encoder, batch_size=batch_size, device=device)
    
    # Encode human texts
    human_data = encode_batch(human_texts, encoder, batch_size=batch_size, device=device)
    
    with open(path, "wb") as f:
        pickle.dump((human_data, ai_data), f)

In [5]:
from local_utilities.dataset import get_dataset_train, get_dataset_validation, get_dataset_test
human_train, ai_train, _ = get_dataset_train(randomize=False)
human_val, ai_val, _ = get_dataset_validation(randomize=False)
human_test, ai_test, _ = get_dataset_test(randomize=False)

In [6]:
# Encode train data
from local_utilities.directory import get_sbert_path_train
encode_dataset(human_train, ai_train, get_sbert_path_train(), encoder,
               batch_size=8, device=device)

100%|██████████| 60011/60011 [06:26<00:00, 155.09it/s]


In [7]:
# Encode validation data
from local_utilities.directory import get_sbert_path_validate
encode_dataset(human_val, ai_val, get_sbert_path_validate(), encoder,
               batch_size=8, device=device)

100%|██████████| 12860/12860 [01:21<00:00, 156.96it/s]


In [8]:
# Encode test data
from local_utilities.directory import get_sbert_path_test
encode_dataset(human_test, ai_test, get_sbert_path_test(), encoder,
               batch_size=8, device=device)

100%|██████████| 12860/12860 [01:21<00:00, 157.15it/s]
